# Class 4 Question Bank Quality Audit

## tl;dr

The current bank is not ready to be treated as a verified, official-source-only assessment. It contains 456 rows but only 402 unique IDs, has no page-level citations, omits required Class 4 study areas, and contains strong answer-pattern cues. The automated checks below are deterministic; factual correctness still requires the documented manual-review gate.

## Context & Methods

This notebook audits the shipped `questions.js` file at one-row-per-question grain. Checks cover schema integrity, ID uniqueness, exact duplicates, image existence, chapter/class coverage, source labeling, explanation length, answer-position bias, and answer-length cues.

### Key Assumptions

- `questions.js` is the production question bank.
- The official ICBC *Driving Commercial Vehicles* guide is the controlling study source.
- A source label without document version, page, and supporting excerpt is not an auditable citation.
- Automated checks identify risk but cannot certify every answer as legally current.

In [1]:
from pathlib import Path
import sys
import pandas as pd

display = print
ROOT = Path.cwd().resolve()
if not (ROOT / 'questions.js').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'audit'))
from question_bank_audit import load_question_bank, audit_question_bank

bank = load_question_bank(ROOT / 'questions.js')
audit = audit_question_bank(bank, ROOT)
len(bank)

456

## Data

The profile below defines the audited population and its intended grain.

In [2]:
pd.DataFrame([audit['population']]).T.rename(columns={0: 'value'})

                       value
rows                     456
unique_ids               402
min_id                     1
max_id                   404
questions_with_images    117
unique_images            117

## Results

### Integrity failures

Duplicate IDs are release-blocking because the application keys progress and the mistakes book by question ID.

In [3]:
pd.Series({
    'duplicate ID values': audit['integrity']['duplicate_id_values'],
    'extra rows sharing an ID': audit['integrity']['duplicate_id_extra_rows'],
    'exact duplicate question groups': audit['integrity']['exact_duplicate_question_groups'],
    'invalid option schemas': len(audit['integrity']['invalid_options']),
    'missing image files': len(audit['integrity']['missing_images']),
}, name='count').to_frame()

                                 count
duplicate ID values                 54
extra rows sharing an ID            54
exact duplicate question groups      1
invalid option schemas               0
missing image files                  0

### Answer-key and wording cues

A defensible multiple-choice bank should not reward a fixed letter or the longest option. These statistics do not prove a question is wrong; they show the assessment can be gamed.

In [4]:
answer_rows = [
    {'answer': key, 'count': value, 'share': value / len(bank)}
    for key, value in audit['assessment_signals']['answer_counts'].items()
]
display(pd.DataFrame(answer_rows))
pd.Series({
    'always choose B': audit['assessment_signals']['always_choose_B_score'],
    'longest option, random tie-break': audit['assessment_signals']['longest_option_expected_score_with_random_ties'],
    'correct answer is among longest': audit['assessment_signals']['correct_is_a_longest_option_rate'],
}, name='expected score').to_frame().map(lambda value: f'{value:.1%}')

  answer  count     share
0      A     61  0.133772
1      B    236  0.517544
2      C    137  0.300439
3      D     22  0.048246


                                 expected score
always choose B                           51.8%
longest option, random tie-break          64.0%
correct answer is among longest           79.4%

### Coverage and provenance

The chapter profile shows concentration rather than a blueprint-driven distribution. No question has a page-level citation.

In [5]:
display(pd.DataFrame(audit['coverage']['chapter_profile']))
pd.Series(audit['provenance'], name='count').to_frame()

     chapter  questions  with_image  ...  answer_C  answer_D  answer_B_rate
0   chapter1          7           0  ...         2         1       0.428571
1  chapter10         61           0  ...         9         1       0.672131
2  chapter11        117         117  ...        38        11       0.418803
3   chapter2         68           0  ...        20         4       0.514706
4   chapter3         64           0  ...        18         2       0.531250
5   chapter6         85           0  ...        33         2       0.505882
6   chapter7         54           0  ...        17         1       0.574074

[7 rows x 8 columns]


                          count
with_source_label           377
without_source_label         79
with_page_citation            0
source_only_explanations     10

### High-risk manual-review sample

These examples were checked against the supplied official manual text or against the displayed image. They are examples, not a claim that every unlisted question is valid.

In [6]:
findings = pd.read_csv(ROOT / 'audit' / 'manual_review_findings.csv')
display(findings)

   severity  ...                                 evidence_or_action
0  Critical  ...  Renumber all rows with stable immutable IDs be...
1      High  ...  Specify the vehicle type; the manual separatel...
2      High  ...  Re-source from an official ICBC or RoadSafetyB...
3      High  ...  Rewrite the stem to include all conditions lis...
4      High  ...  Verify against a current official municipal or...
5      High  ...  Replace with the manual's own blind-spot termi...
6      High  ...  Remove the exact interval unless an official s...
7    Medium  ...  Keep one item and cite the exact current offic...
8    Medium  ...  Use an unambiguous right-arrow asset or ask on...
9    Medium  ...  Use one comparison item or differentiate the s...

[10 rows x 6 columns]


## Takeaways

1. Fix ID collisions before further practice data is collected.
2. Replace prose source labels with structured, versioned page citations and excerpts.
3. Build the bank from an official study blueprint, including the missing required areas.
4. Regenerate distractors and shuffle answer positions under explicit balance constraints.
5. Require automated checks plus two-stage factual/editorial review before release.

The notebook validates structure and assessment signals. It does not certify that all 456 answers are legally current; that requires a question-by-question source review against a pinned official manual version and current regulations.